# 04 · Reglas de asociación (CRISP-DM: Modelado)

Se identifican productos que suelen comprarse juntos (análisis de canasta con FP-Growth), para dar sustento a las recomendaciones de productos propuestas para los segmentos "Recientes ocasionales" y "En riesgo". Se analizan las facturas del Reino Unido (85 % de las ventas).

In [1]:
import sqlite3
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

conn = sqlite3.connect("../data/processed/retail.db")
ventas = pd.read_sql("""
    SELECT invoice, stock_code, description
    FROM ventas_limpias
    WHERE country = 'United Kingdom'
""", conn)

n_facturas = ventas["invoice"].nunique()

# Solo productos presentes en al menos 1 % de las facturas (reduce memoria y ruido)
soporte_producto = ventas.groupby("stock_code")["invoice"].nunique() / n_facturas
productos = soporte_producto[soporte_producto >= 0.01].index
sub = ventas[ventas["stock_code"].isin(productos)]

# Matriz factura x producto (True si el producto está en la factura)
canastas = pd.crosstab(sub["invoice"], sub["stock_code"]).gt(0)
canastas = canastas.reindex(ventas["invoice"].unique(), fill_value=False)

print("Facturas:", n_facturas, "| Productos considerados:", len(productos))

Facturas: 33361 | Productos considerados: 552


In [2]:
frecuentes = fpgrowth(canastas, min_support=0.02, use_colnames=True)
reglas = association_rules(frecuentes, num_itemsets=len(canastas),
                           metric="lift", min_threshold=1.5)

# Código -> nombre del producto (el nombre más usado para cada código)
nombres = (ventas.dropna(subset=["description"])
                 .groupby("stock_code")["description"]
                 .agg(lambda s: s.mode().iat[0]))

def a_texto(items):
    return " + ".join(nombres.get(c, c) for c in items)

reglas["si_compra"] = reglas["antecedents"].apply(a_texto)
reglas["tambien_compra"] = reglas["consequents"].apply(a_texto)

tabla = (reglas[reglas["confidence"] >= 0.5]
         .sort_values("lift", ascending=False)
         [["si_compra", "tambien_compra", "support", "confidence", "lift"]])

tabla.to_csv("../data/processed/reglas_asociacion.csv", index=False)
print("Reglas encontradas:", len(tabla))
tabla.head(15).round(3)

Reglas encontradas: 14


,si_compra,tambien_compra,support,confidence,lift
21,SWEETHEART CERAMIC TRINKET BOX,STRAWBERRY CERAMIC TRINKET BOX,0.024,0.722,14.020
29,WOODEN FRAME ANTIQUE WHITE,WOODEN PICTURE FRAME WHITE FINISH,0.029,0.565,11.709
28,WOODEN PICTURE FRAME WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.029,0.603,11.709
1,LOVE BUILDING BLOCK WORD,HOME BUILDING BLOCK WORD,0.023,0.528,9.970
19,PACK OF 60 PINK PAISLEY CAKE CASES,PACK OF 72 RETROSPOT CAKE CASES,0.022,0.529,8.341
17,60 TEATIME FAIRY CAKE CASES,PACK OF 72 RETROSPOT CAKE CASES,0.024,0.514,8.105
15,LUNCH BAG WOODLAND,LUNCH BAG RED RETROSPOT,0.022,0.553,8.071
43,LUNCH BAG PINK POLKADOT,LUNCH BAG RED RETROSPOT,0.028,0.548,8.009
25,LUNCH BAG BLACK SKULL.,LUNCH BAG RED RETROSPOT,0.029,0.506,7.388
31,JUMBO BAG STRAWBERRY,JUMBO BAG RED RETROSPOT,0.027,0.633,7.067


## Conclusiones

- Con soporte mínimo de 2 %, lift ≥ 1.5 y confianza ≥ 50 % se obtuvieron 14 reglas, con lift entre 5 y 14.
- La mayoría relaciona el mismo artículo en distintos diseños (bolsas de almuerzo, bolsas jumbo, moldes para cupcakes, portavelas): los clientes compran colecciones.
- El diseño "Red Retrospot" actúa como producto ancla: es el consecuente en 8 de las 14 reglas.
- Aplicación a los segmentos: recomendar los otros diseños de la colección a los "Recientes ocasionales" y ofrecer packs de colección a los clientes "En riesgo". Para inventario, mantener stock de los productos ancla.
- Limitación: el soporte de las reglas es bajo (2–3 % de las facturas), por lo que cubren una parte acotada de las compras.
- Resultado exportado a `data/processed/reglas_asociacion.csv`.